<a href="https://colab.research.google.com/github/gilda14/MediVision-AI/blob/main/MediVision_AI_Lesson_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

patient = {
    "name": "Test Patient",
    "age": 45,
    "glucose": 5.8,
    "cholesterol": 4.2
}

# MediVision AI - Check Medical Test Result

glucose = patient["glucose"]

minimum = 3.9
maximum = 5.5

if glucose < minimum:
    print("Glucose is BELOW the example range")

elif glucose > maximum:
    print("Glucose is ABOVE the example range")

else:
    print("Glucose is WITHIN the example range")



Glucose is ABOVE the example range


In [ ]:

from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("OpenAI client created successfully!")


OpenAI client created successfully!


In [ ]:

# MediVision AI - First LLM Request

response = client.responses.create(
    model="gpt-4.1-mini",
    input="""
    Summarise this fictional patient note
    in one sentence. Do not diagnose.

    Patient is 45 years old.
    Blood pressure is 120/80 mmHg.
    Heart rate is 72 bpm.
    """
)

print(response.output_text)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-*******************************************************************************************************************************************************dx04. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:

from google.colab import userdata
from openai import OpenAI

# Load the new API key
api_key = userdata.get("OPENAI_API_KEY")

# Create a new connection
client = OpenAI(api_key=api_key)

print("New OpenAI client created successfully!")


New OpenAI client created successfully!


In [ ]:

# MediVision AI - Test AI Connection

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say Hello! I am MediVision AI."
)

print(response.output_text)


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:

# MediVision AI - Test AI Connection

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say Hello! I am MediVision AI."
)

print(response.output_text)


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:

!pip install openai


Import the OpenAI library

In [ ]:

from openai import OpenAI

print("OpenAI library loaded successfully!")


OpenAI library loaded successfully!


In [ ]:

# MediVision AI - Install Free LLM Libraries

!pip install -q transformers accelerate sentencepiece


** Load your first free AI model**    
What does this code mean?
- pipeline provides a simple way to use an AI model.
- google/flan-t5-small identifies the model we want to download.
- device=0 tells Python to use your first GPU.

In [ ]:


# MediVision AI - Load Free LLM

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("MediVision AI model loaded successfully!")
print("Running on:", device)



tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

MediVision AI model loaded successfully!
Running on: cpu


In [ ]:

# MediVision AI - First Medical Report Summary

medical_report = """
Patient: Test Patient
Age: 45
Blood Pressure: 120/80 mmHg
Heart Rate: 72 bpm
Glucose: 5.8 mmol/L
Cholesterol: 4.2 mmol/L
"""

prompt = (
    "Summarize the following patient report. "
    "Only use the information provided. "
    "Do not diagnose.\n"
    + medical_report
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("AI Medical Report Summary:")
print(summary)


AI Medical Report Summary:
                                                 


In [ ]:

# MediVision AI - Improve Medical Summary

prompt = """
Summarize this medical information in one sentence:

A 45-year-old patient has a blood pressure
of 120/80 mmHg and a heart rate of 72 bpm.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    num_beams=4
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("AI Summary:", repr(summary))


AI Summary: 'The patient has a heart rate of 120/80 mmHg and a heart rate of 72 bpm.'


We're beginning to build an AI quality-checking system.

This simple test checks whether important terms appear in the generated summary. It does not yet verify that the medical facts are correctly associated with their measurements

In [ ]:

# MediVision AI - Check Summary Accuracy

expected_terms = [
    "blood pressure",
    "120/80",
    "heart rate",
    "72"
]

for term in expected_terms:
    if term.lower() in summary.lower():
        print("FOUND:", term)
    else:
        print("MISSING:", term)


MISSING: blood pressure
FOUND: 120/80
FOUND: heart rate
FOUND: 72


In [7]:

# MediVision AI - Check Medical Measurements

summary = (
    "The patient has a heart rate of 120/80 mmHg "
    "and a heart rate of 72 bpm."
)

# Check whether the summary contains
# the correct blood pressure information

if "blood pressure of 120/80" in summary.lower():
    print("PASS: Blood pressure is correctly described")
else:
    print("WARNING: Blood pressure information may be incorrect")


**Check both correct and incorrect AI summaries**

def means we're defining a function.

1.  check_medical_summary is the function's name.
2.   summary is the information we pass into the function.
1.  The indented code runs whenever we call the function.


In [1]:

# MediVision AI - Medical Summary Validator

def check_medical_summary(summary):

    if "blood pressure of 120/80" in summary.lower():
        print("PASS: Blood pressure is correctly described")
    else:
        print("WARNING: Blood pressure may be incorrect")


# Test 1 - Correct summary
summary1 = "The patient has a blood pressure of 120/80 mmHg."

print("Test 1:")
check_medical_summary(summary1)


# Test 2 - Incorrect summary
summary2 = "The patient has a heart rate of 120/80 mmHg."

print("Test 2:")
check_medical_summary(summary2)


Test 1:
PASS: Blood pressure is correctly described
Test 2:


In [2]:

# MediVision AI - Extract Patient Information

# Fictional medical report
medical_report = """
Patient Name: John Smith
Age: 45
Blood Pressure: 120/80 mmHg
Heart Rate: 72 bpm
Glucose: 5.8 mmol/L
"""

# Create an empty dictionary
patient_data = {}

# Read the report line by line
for line in medical_report.strip().split("\n"):

    # Separate the field name and its value
    #strip()	Removes unnecessary spaces
    key, value = line.split(":", 1)

    # Store the information and Saves each piece of information
    patient_data[key.strip()] = value.strip()

# Display the extracted information
print("Extracted Patient Information:")

for key, value in patient_data.items():
    print(key, ":", value)





Extracted Patient Information:
Patient Name : John Smith
Age : 45
Blood Pressure : 120/80 mmHg
Heart Rate : 72 bpm
Glucose : 5.8 mmol/L
